In [1]:
import pandas as pd
import numpy as np
import sys
from copy import deepcopy,copy
from datetime import datetime
import pickle
import sys
import matplotlib.pyplot as plt
import pynumdiff
import traceback
import time
import os
# Import Machine Scientist
from importlib.machinery import SourceFileLoader
# Get the absolute path of the script's directory
script_dir = '/export/home/oriolca/Integral_BMS_Governing_Equations/strogatz/'
# Define the relative path to the module
relative_module_path = "../I-BMS-2d/parallel_ode.py"
path = os.path.join(script_dir, relative_module_path)
ms = SourceFileLoader("ms", path).load_module()

#Import prior
path = os.path.join(script_dir, "../I-BMS-2d/Prior/")
sys.path.append(path)
from fit_prior import read_prior_par
prior = read_prior_par('../I-BMS-2d/Prior/final_prior_param_sq.named_equations.nv2.np8.2016-09-09 18:49:42.800618.dat')

2025-10-24 17:53:09,604 [INFO] 
Limited Total Variation Regularization Support Detected! 
---> CVXPY is not installed. 
---> Many Total Variation Methods require CVXPY including: 
---> velocity, acceleration, jerk, jerk_sliding, smooth_acceleration
---> Please install CVXPY to use these methods.
---> Recommended to also install MOSEK and obtain a MOSEK license.
You can still use: total_variation_regularization.iterative_velocity

2025-10-24 17:53:09,605 [INFO] 
Limited Linear Model Support Detected! 
---> PYCHEBFUN is not installed. 
---> Install pychebfun to use chebfun derivatives (https://github.com/pychebfun/pychebfun/) 
You can still use other methods 

2025-10-24 17:53:09,605 [INFO] 
Limited Linear Model Support Detected! 
---> CVXPY is not installed. 
---> Install CVXPY to use lineardiff derivatives 
You can still use other methods 



In [2]:
data=pd.read_csv('datasets/bact_res_SNR_50_0.csv')

x={}
y={}
dx = {}

dy = {}

x['d0']=deepcopy(data)
y['d0']=deepcopy(data)
y['d0'].x=deepcopy(x['d0'].y)
y['d0'].y=deepcopy(x['d0'].x)

h = x['d0'].t.to_numpy()[1] - x['d0'].t.to_numpy()[0]

par = [2, 21, 21]
x_hat, dxdt_hat = pynumdiff.linear_model.polydiff(
        x['d0'].x, h, par, options=None)
y_hat, dydt_hat = pynumdiff.linear_model.polydiff(
        x['d0'].y, h, par, options=None)

dx['d0'] = [ pd.DataFrame(data={'x':x_hat,'y':y_hat}),
            pd.DataFrame(data={'x':dxdt_hat,'y':dydt_hat})]

dy['d0'] = [ pd.DataFrame(data={'x':y_hat,'y':x_hat}),
            pd.DataFrame(data={'x':dydt_hat,'y':dxdt_hat})]


mcmc_resets = 2
mcmc_steps = 30
XLABS = ['x','y']
params = 8
print(x)
print(y)

description_lengths, mdl, mdl_x, mdl_y, mdl_model_x, mdl_model_y = (
        [],
        np.inf,
        np.inf,
        np.inf,
        None,
        None,
    )

del prior["Nopi_abs"]
del prior["Nopi2_abs"]
del prior["Nopi_sin"]
del prior["Nopi2_sin"]
del prior["Nopi_cos"]
del prior["Nopi2_cos"]
del prior["Nopi_tan"]
del prior["Nopi2_tan"]
del prior["Nopi_sinh"]
del prior["Nopi2_sinh"]
del prior["Nopi_cosh"]
del prior["Nopi2_cosh"]
del prior["Nopi_tanh"]
del prior["Nopi2_tanh"]
OPS = {
    #'sin': 1,
    #'cos': 1,
    #'tan': 1,
    "exp": 1,
    #'log': 1,
    #'sinh' : 1,
    #'cosh' : 1,
    #'tanh' : 1,
    "pow2": 1,
    "pow3": 1,
    #'sqrt' : 1,
    #'fac' : 1,
    "-": 1,
    "+": 2,
    "*": 2,
    "/": 2,
    "**": 2,
}
#stderr_fileno = sys.stderr
#sys.stderr = open(os.devnull, 'w')
from multiprocessing import Pool

def run_instance(_):
    dl, i_mdl, model_x, model_y = [], np.inf, None, None
    Ts=[1] + [1.04**k for k in range(1, 40,2)]
    pms_x = ms.Parallel(
        Ts,
        ops=OPS,
        variables=XLABS,
        parameters=["a%d" % i for i in range(params)],
        x=x,
        dx=dx,
        prior_par=prior,
    )
    pms_y = ms.Parallel(
        Ts,
        ops=OPS,
        variables=XLABS,
        parameters=["a%d" % i for i in range(params)],
        x=y,
        dx=dy,
        prior_par=prior,
    )
    print("setting f-g links")
    for temp in pms_x.trees.keys():
        pms_x.trees[temp].fy = pms_y.trees[temp]
        pms_y.trees[temp].fy = pms_x.trees[temp]
        # print('refit')
        pms_x.trees[temp].get_bic(reset=True, fit=True)
        pms_x.trees[temp].get_energy(bic=True, reset=True)
    pms_x.t1 = pms_x.trees[str(min(Ts))]
    pms_y.t1 = pms_y.trees[str(min(Ts))]
    print('Initial MCMC model x:',pms_x.t1,pms_x.t1.E)
    print('Initial MCMC model y:',pms_y.t1,pms_y.t1.E)
    mc_start = time.time()
    for i in range(1, mcmc_steps + 1):
        start = time.time()
        # MCMC update
        pms_x.mcmc_step()  # MCMC step within each T
        pms_y.mcmc_step()
        ET1, ET2 = (
            pms_x.tree_swap()
        )  # Attempt to swap two randomly selected consecutive temps

        if ET1 != None:
            t1 = pms_y.trees[ET1]
            t2 = pms_y.trees[ET2]
            BT1, BT2 = t1.BT, t2.BT
            pms_y.trees[ET1] = t2
            pms_y.trees[ET2] = t1
            t1.BT = BT2
            t2.BT = BT1
            pms_x.trees[ET1].get_bic(reset=True, fit=True)
            pms_x.trees[ET1].get_energy(bic=False, reset=True)
            pms_y.t1 = pms_y.trees[str(min(Ts))]

        dl.append(copy(pms_x.t1.E ))
        # Add the description length to the trace
        # description_lengths.append(pms.t1.E)
        # Check if this is the MDL expression so far
        if pms_x.t1.E < i_mdl:
            # if pms.t1.E==float('NaN'): print('NaN in best model mdl')
            i_mdl = copy(pms_x.t1.E)
            model_x = deepcopy(pms_x.t1)
            model_y = deepcopy(pms_y.t1)
    return dl,i_mdl,model_x,model_y
    
with Pool(processes=2, maxtasksperchild=1) as pool:
    results = pool.map(run_instance, range(2))
for result in results:
    #dl, , smooth_E, combo_x, combo_y = result
    print('parallel res:',result[1:])
    description_lengths.append(result[0])
    if result[1] <mdl:
        description_lengths.append(result[0])
        mdl=copy(result[1])
        mdl_model_x = deepcopy(result[2])
        mdl_model_y = deepcopy(result[3])

{'d0':       Unnamed: 0      t         x          y
0              0   0.00  9.023431  12.803829
1              1   0.01  9.089200  12.845921
2              2   0.02  9.171832  12.907267
3              3   0.03  9.255938  13.039091
4              4   0.04  9.347307  13.097073
...          ...    ...       ...        ...
2995        2995  29.95  9.799593  50.841826
2996        2996  29.96  9.803124  50.834549
2997        2997  29.97  9.809116  50.856895
2998        2998  29.98  9.801152  50.854903
2999        2999  29.99  9.813299  50.807510

[3000 rows x 4 columns]}
{'d0':       Unnamed: 0      t          x         y
0              0   0.00  12.803829  9.023431
1              1   0.01  12.845921  9.089200
2              2   0.02  12.907267  9.171832
3              3   0.03  13.039091  9.255938
4              4   0.04  13.097073  9.347307
...          ...    ...        ...       ...
2995        2995  29.95  50.841826  9.799593
2996        2996  29.96  50.834549  9.803124
2997        299

<lambdifygenerated-605>:2: RuntimeWarning: invalid value encountered in scalar power
  return _a0_**_a7_
/home/oriolca/.local/lib/python3.10/site-packages/scipy/optimize/_minpack_py.py:1010: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',
<lambdifygenerated-837>:2: RuntimeWarning: invalid value encountered in scalar power
  return _a3_**_a2_
/home/oriolca/.local/lib/python3.10/site-packages/scipy/optimize/_optimize.py:2489: RuntimeWarning: invalid value encountered in scalar multiply
  tmp2 = (x - v) * (fx - fw)
<lambdifygenerated-913>:2: RuntimeWarning: invalid value encountered in scalar power
  return _a7_**_a1_
<lambdifygenerated-977>:2: RuntimeWarning: overflow encountered in scalar power
  return x**3
/export/home/oriolca/Integral_BMS_Governing_Equations/I-BMS-2d/mcmc_ode.py:810: RuntimeWarning: overflow encountered in square
  sse = np.sum(residuals ** 2.)
/home/oriolca/.local/lib/python3

parallel res: (14289.6363695837, (x * _a4_), (_a0_ + y))
parallel res: (14524.3655712348, _a0_, (_a0_ + y))


In [3]:
print(mdl_model_x)
print(mdl_model_y)

(x * _a4_)
(_a0_ + y)
